# GHS Survey Data

Data downloaded on 3/18/26 from this survey we put together: https://docs.google.com/forms/d/1YcFjLg2toGALK49KQOwk12ZU4U4yBCwtdAHTGF6R88E

In [ ]:
# TODO: change survey to have dropdown of all NYC schools if possible
# TODO: change survey to have multiple choice for grade level
# TODO: change survey to have multiple choice for "how much you care" and then maybe follow with "what have you done to try to fix it" or something

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
survey_df = pd.read_csv("../data/raw_data/GHS_Survey/Green and Healthy Schools Survey (Responses) - Form Responses 2026-03-18.csv")
renamed_cols = {
    "Name (first and last name)": "name",
    "School Name": "school_raw",
    "Neighborhood of school": "neighborhood",
    "Grade (if student) or Job Title (if teacher/staff)": "grade_raw",
    "1. If you could fix 1 problem with your school building, what would it be?": "q1_one_problem_to_fix",
    "2. Rate the following things at your school [Heating]": "q2_heating",
    "2. Rate the following things at your school [Air Conditioning]": "q2_air_conditioning",
    "2. Rate the following things at your school [Drinking Water]": "q2_drinking_water",
    "2. Rate the following things at your school [Classrooms]": "q2_classrooms",
    "2. Rate the following things at your school [Bathrooms]": "q2_bathrooms",
    "2. Rate the following things at your school [Cafeteria]": "q2_cafeteria",
    "2. Rate the following things at your school [Gym]": "q2_gym",
    "2. Rate the following things at your school [Playground / Outdoor Space]": "q2_playground_outdoor_space",
    "2. Rate the following things at your school [Teachers' Lounge / Offices]": "q2_teachers_lounge_offices",
    "3. Is there any active construction happening at your school?": "q3_active_construction",
    "If so, tell us more (e.g., how long has it been going on, what is the purpose of the construction, have there been any problems) ": "q3_construction_details",
    "4. Does your school experience any of the following? [Flooding]": "q4_flooding",
    "4. Does your school experience any of the following? [Power Outages]": "q4_power_outages",
    "4. Does your school experience any of the following? [Extremely Hot Indoors]": "q4_extremely_hot_indoors",
    "4. Does your school experience any of the following? [Extremely Cold Indoors]": "q4_extremely_cold_indoors",
    "4. Does your school experience any of the following? [Outdoor Area / Equipment Gets too Hot]": "q4_outdoor_area_too_hot",
    "4. Does your school experience any of the following? [Toxins Present (lead, asbestos, etc.)]": "q4_toxins_present",
    "4. Does your school experience any of the following? [Climate Change]": "q4_climate_change",
    "5. How is your school community fighting climate change?": "q5_fighting_climate_change",
    "6. Tell us more about any of the issues mentioned in questions 1-4": "q6_issue_details",
    "7. Who do you think has the power to fix the problems at your school?": "q7_power_to_fix",
    "8. Why do you think these problems haven’t been fixed?": "q8_problems_not_fixed",
    "9. How much do you care about these problems?": "q9_care_about_problems",
    "10. Are you interested in getting more involved?": "q10_get_involved"
}
survey_df = survey_df.rename(columns=renamed_cols)

Turn categorical answers into numeric to make analysis easier
- 1 - Very Bad
- 2 - Bad
- 3 - Okay
- 4 - Good
- 5 - Very Good

In [ ]:
q2_cols = [col for col in survey_df.columns if col.startswith('q2_')]
survey_df[q2_cols] = survey_df[q2_cols].replace({
    "1 - Very Bad": 1,
    "2 - Bad": 2,
    "3 - Okay": 3,
    "4 - Good": 4,
    "5 - Very Good": 5
})

In [ ]:
q4_cols = [col for col in survey_df.columns if col.startswith('q4_')]
survey_df[q4_cols] = survey_df[q4_cols].replace({'Yes': 1, 'No': 0})

survey_df['q10_get_involved'] = survey_df['q10_get_involved'].replace({'Yes': 1, 'No': 0})

Clean `grade`

In [ ]:
survey_df['grade_clean'] = survey_df['grade_raw'].str.lower().str.strip()

survey_df['grade_clean'] = survey_df['grade_clean'].str.replace(r'year 1', 'College')

survey_df['grade_clean'] = survey_df['grade_clean'].str.extract(r'(\d+)', expand=False).combine_first(survey_df['grade_clean'])

# TODO: this is coercing freshman/sophomore/junior/senior for college students into high school grades; need to fix
freshman_mask = survey_df['grade_clean'].fillna('').str.contains(r'freshman|nine|ninth', regex=True)
sophomore_mask = survey_df['grade_clean'].fillna('').str.contains(r'sophomore|ten|tenth', regex=True)
junior_mask = survey_df['grade_clean'].fillna('').str.contains(r'junior|eleven|eleventh', regex=True)
senior_mask = survey_df['grade_clean'].fillna('').str.contains(r'senior|twelve|twelf?th', regex=True)
college_mask = survey_df['grade_clean'].fillna('').str.contains('college')

survey_df.loc[freshman_mask, 'grade_clean'] = '9'
survey_df.loc[sophomore_mask, 'grade_clean'] = '10'
survey_df.loc[junior_mask, 'grade_clean'] = '11'
survey_df.loc[senior_mask, 'grade_clean'] = '12'
survey_df.loc[college_mask, 'grade_clean'] = 'College'

# survey_df[['grade_raw', 'grade_clean']].drop_duplicates().sort_values('grade_clean')

Clean school names

In [ ]:
survey_df['school_clean'] = survey_df['school_raw'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)

# NOTE: there are 3 different "Abraham Lincoln" schools in NYC: "Abraham Lincoln High School", "I.S. 171 Abraham Lincoln", and "P.S. 007 Abraham Lincoln"
# All of the students that took this survey on 3/17/2026 are assumed to be in the high school, so the below regex reflects that (will need to be adjusted if we get surveys from elementary or middle school students in the future)
alhs_mask = (
    survey_df['school_clean'].str.contains(r'abraham.*[kl]inc?|alhs', regex=True)
    |
    ( survey_df['school_clean'].str.contains(r'lincoln') & survey_df['neighborhood'].str.lower().str.contains(r'coney|brighton', regex=True) )
)
survey_df.loc[alhs_mask, 'school_clean'] = 'Abraham Lincoln High School'

# NOTE: there is a Bard HS Early College in Manhattan, Brooklyn, Queens, and the Bronx. The Manhattan one is just called "Bard High School Early College"; the rest have the borough name at the end of the name
bard_hs_mask = (
    survey_df['school_clean'].str.contains(r'bard.*queens|bhsecq|bhsec.*queen', regex=True)
    |
    ( survey_df['school_clean'].str.contains(r'bard') & survey_df['neighborhood'].str.lower().str.contains(r'queens|lic|long island city|sunnyside', regex=True) )
)
survey_df.loc[bard_hs_mask, 'school_clean'] = 'Bard High School Early College Queens'

# NOTE: there is "Beacon High School" and also "P.S. 172 Beacon School of Excellence". As of 3/17/26 we're assuming all "beacon" schools are the high school, but this will need to be adjusted if we get surveys from elementary school students in the future
beacon_hs_mask = survey_df['school_clean'].str.contains(r'beacon', regex=True)
survey_df.loc[beacon_hs_mask, 'school_clean'] = 'Beacon High School'

# NOTE: There is "Brooklyn College Academy", which is a HS, and then actual Brooklyn College (CUNY). Manually reviewed the 2 BKC students and they're both "seniors", so I think it's HS.
brooklyn_college_mask = survey_df['school_clean'].str.contains(r'brool?klyn.*college', regex=True)
survey_df.loc[brooklyn_college_mask, 'school_clean'] = 'Brooklyn College Academy'

brooklyn_tech_mask = survey_df['school_clean'].str.contains(r'brooklyn.*tech', regex=True)
survey_df.loc[brooklyn_tech_mask, 'school_clean'] = 'Brooklyn Technical High School'

csi_mask = survey_df['school_clean'].str.contains('csi')
survey_df.loc[csi_mask, 'school_clean'] = 'CSI High School for International Studies'

curtis_hs_mask = survey_df['school_clean'].str.contains(r'curtis high', regex=True)
survey_df.loc[curtis_hs_mask, 'school_clean'] = 'Curtis High School'

# NOTE: there's a "Fannie Lou..." HS and MS. Assuming the 3/17/26 survey is all HS (since only two 8th graders and neither list fannie lou as school)
fannie_lou_hs_mask = survey_df['school_clean'].str.contains(r'fannie.?l[io]u', regex=True)
survey_df.loc[fannie_lou_hs_mask, 'school_clean'] = 'Fannie Lou Hamer Freedom High School'

# NOTE: there's a laguardia HS of performing arts and then also "International High School at LaGuardia Community College".
laguardia_cc_mask = survey_df['school_clean'].str.contains(r'laguardia.*community.*college|laguardia.*cc', regex=True)
survey_df.loc[laguardia_cc_mask, 'school_clean'] = 'International High School at LaGuardia Community College'

# NOTE: ORDER MATTERS HERE. Doing CC first and the case sensitivity will make sure the LGA CC match doesn't trigger the LGA performing arts regex
laguardia_hs_mask = survey_df['school_clean'].str.contains(r'laguardia|lagaurdia', regex=True)
survey_df.loc[laguardia_hs_mask, 'school_clean'] = 'Fiorello H. LaGuardia High School of Music & Art and Performing Arts'

# NOTE: there is both a HS and elementary school named "Francis Lewis". Assuming the 3/17/26 survey is all HS
francis_lewis_mask = survey_df['school_clean'].str.contains(r'francis.*lewis', regex=True)
survey_df.loc[francis_lewis_mask, 'school_clean'] = 'Francis Lewis High School'

hsmse_ccny_mask = survey_df['school_clean'].str.contains(r'high school for math|hsmse', regex=True)
survey_df.loc[hsmse_ccny_mask, 'school_clean'] = 'High School for Mathematics, Science and Engineering at City College'

# NOTE: there is a John Jay School for Law HS in Brooklyn, but the 2 surveys we have from 3/17/26 are both from college students. Need to update if we have any
john_jay_college_mask = survey_df['school_clean'].str.contains(r'john jay college', regex=True)
survey_df.loc[john_jay_college_mask, 'school_clean'] = 'John Jay College of Criminal Justice'

# NOTE: I can't manually figure out which school in the LCGMS data matches here, but assuming it's the school(s) associated with https://kippnyccphs.org/
# Based on https://kippnyc.org/schools/ there's only one KIPP HS in NYC but many middle/elementary, so if we get younger KIPP students taking the survey, this will get a lot harder, but at least for the 3/17/26 results this is good enough
kipp_nyc_mask = survey_df['school_clean'].str.contains(r'kipp.?nyc|kipp.*college|kipp.*high', regex=True)
survey_df.loc[kipp_nyc_mask, 'school_clean'] = 'KIPP NYC College Prep High School'

# See https://ms50.org/whoweare for more. This is the one partnered with El Puente, but in LCGMS data it's "J.H.S. 050 John D. Wells"
ms50_mask = survey_df['school_clean'].str.contains(r'ms.*50', regex=True)
survey_df.loc[ms50_mask, 'school_clean'] = 'J.H.S. 050 John D. Wells'

nestm_mask = survey_df['school_clean'].str.contains(r'nest.?m|new explor', regex=True)
survey_df.loc[nestm_mask, 'school_clean'] = 'New Explorations into Science, Technology and Math High School'

# NOTE: there is both a "New Heights" MS and HS, both charter 
new_heights_hs_charter_mask = survey_df['school_clean'].str.contains(r'new heights acad', regex=True)
survey_df.loc[new_heights_hs_charter_mask, 'school_clean'] = 'New Heights Academy Charter School'

newtown_mask = survey_df['school_clean'].str.contains(r'newtown', regex=True)
survey_df.loc[newtown_mask, 'school_clean'] = 'Newtown High School'

# NOTE: similar to Bard, there's multiple YWLS around the city, distinguished by location (there's both Queens and Astoria for some reason). 
# Assuming for now that all our 3/17/26 survey takers are from TYWLS Astoria, but if we get more survey data in the future we'll likely need to adjust this and add a new column for borough or something
tywls_mask = (
    survey_df['school_clean'].str.contains(r'tywls.*astoria|young womens leadership.*astoria', regex=True)
    |
    (survey_df['school_clean'].str.contains(r'tywls') & survey_df['neighborhood'].str.lower().str.contains(r'astoria', regex=True))
)
survey_df.loc[tywls_mask, 'school_clean'] = "Young Women's Leadership School, Astoria"

townsend_harris_mask = survey_df['school_clean'].str.contains(r'townsend')
survey_df.loc[townsend_harris_mask, 'school_clean'] = "Townsend Harris High School"

# This is to catch invalid responses in this field, e.g. person put their name instead of school
bad_data_mask = survey_df['school_clean'].str.contains(r'paulina korczakowska|hshshahha', regex=True)
survey_df.loc[bad_data_mask, 'school_clean'] = np.nan

# survey_df[['school_raw', 'school_clean']].drop_duplicates().sort_values('school_clean')

In [ ]:
survey_df['school_clean'].value_counts()

In [ ]:
pd.set_option('display.max_columns', None)
survey_df[survey_df['school_clean']=='Fiorello H. LaGuardia High School of Music & Art and Performing Arts']

In [ ]:
survey_df.groupby('school_clean').agg({
    'name': 'count',
    'q2_heating': 'mean',
    'q2_air_conditioning': 'mean',
    'q2_drinking_water': 'mean',
    'q2_classrooms': 'mean',
    'q2_bathrooms': 'mean',
    'q2_cafeteria': 'mean',
    'q2_gym': 'mean',
    'q2_playground_outdoor_space': 'mean',
    'q2_teachers_lounge_offices': 'mean',
    'q4_flooding': 'mean',
    'q4_power_outages': 'mean',
    'q4_extremely_hot_indoors': 'mean',
    'q4_extremely_cold_indoors': 'mean',
    'q4_outdoor_area_too_hot': 'mean',
    'q4_toxins_present': 'mean',
    'q4_climate_change': 'mean',
    'q10_get_involved': 'sum'
}).reset_index().sort_values('name', ascending=False).rename(columns={'name': 'num_survey_responses'})#.to_csv("../data/processed_data/ghs_survey_data_by_school_pivot_cat_mean.csv", index=False)

In [ ]:
survey_df.to_csv("../data/processed_data/ghs_survey_data_cleaned.csv", index=False)